In [1]:
import os; os.environ['WORKDIR'] = "/home/choij/workspace/ChargedHiggsAnalysis"
import sys; sys.path.insert(0, os.environ['WORKDIR'])

In [8]:
# load files and models
MASSPOINT = "MHc-100_MA-60"
BACKGROUND = "TTLL_powheg"
ERA = "2018"

from ROOT import TFile
f_sig = TFile.Open(f"{os.environ['WORKDIR']}/SelectorOutput/{ERA}/Skim3Mu__/Split/Selector_TTToHcToWAToMuMu_{MASSPOINT}_0.root")
f_bkg = TFile.Open(f"{os.environ['WORKDIR']}/SelectorOutput/{ERA}/Skim3Mu__/Split/Selector_{BACKGROUND}_0.root")

# load model
import torch
from libPython.MLTools import ParticleNet
OPTIM = "AdamW"
InitLR = 0.02
SCHEDULER = "StepLR"
model_path = f"{os.environ['WORKDIR']}/models/pilot/{MASSPOINT}_vs_{BACKGROUND}/ParticleNet_{OPTIM}_initLR-{str(InitLR).replace('.', 'p')}_{SCHEDULER}.pt"
model = ParticleNet(num_features=9, num_classes=2, hidden_channels=128)
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))

<All keys matched successfully>

In [20]:
from libPython.Preprocessor import evt_to_graph
from libPython.Management import predict_proba
def getScore(model, objects):
    node_list = []
    for object in objects:
        node_list.append([object.Pt(),
                         object.Eta(),
                         object.Phi(),
                         object.M(),
                         object.Charge(),
                         object.IsMuon(),
                         object.IsElectron(),
                         object.IsJet(),
                         object.BtagScore()])
    data = evt_to_graph(node_list, y=0, k=4)
    model.eval()
    with torch.no_grad():
        out = model(data.x, data.edge_index)
        print(out)
    #eturn predict_proba(model, data.x, data.edge_index)

In [21]:
from libPython.DataFormat import get_muons, get_electrons, get_jets, Particle
from libPython.Selection import pass_baseline, select
from ROOT import TH1D

h_sig = TH1D("signal", "", 100, 0., 1)
for evt in f_sig.Events:
    muons = get_muons(evt)
    electrons = get_electrons(evt)
    jets, bjets = get_jets(evt)
    METv = Particle(evt.METvPt, 0., evt.METvPhi, 0.)
    
    if not pass_baseline("3Mu", evt, muons, electrons, jets ,bjets, "loose"):
        continue
    #region = select("3Mu", evt, muons, electrons, jets, bjets, "tight")
    #if region != "ZFakeRegion":
    #    continue
     
    score = getScore(model, muons+electrons+jets)
    h_sig.Fill(score)
    

tensor([[0.0013, 0.9987]])


TypeError: none of the 3 overloaded methods succeeded. Full details:
  int TH1::Fill(double x) =>
    TypeError: could not convert argument 1 (must be real number, not NoneType)
  int TH1::Fill(const char* name, double w) =>
    TypeError: takes at least 2 arguments (1 given)
  int TH1::Fill(double x, double w) =>
    TypeError: takes at least 2 arguments (1 given)

Warning in <TFile::Append>: Replacing existing TH1: signal (Potential memory leak).


In [22]:
h_bkg = TH1D("bkg", "", 100, 0., 1)
for evt in f_bkg.Events:
    muons = get_muons(evt)
    electrons = get_electrons(evt)
    jets, bjets = get_jets(evt)
    METv = Particle(evt.METvPt, 0., evt.METvPhi, 0.)
    
    if not pass_baseline("3Mu", evt, muons, electrons, jets, bjets, "loose"):
        continue
    #region = select("3Mu", evt, muons, electrons, jets, bjets, "tight")
    #if region != "ZFakeRegion":
    #    continue
    
    getScore(model, muons+electrons+jets)
    #h_bkg.Fill(score)

tensor([[0.6371, 0.3629]])
tensor([[3.1188e-04, 9.9969e-01]])
tensor([[1.4906e-05, 9.9999e-01]])
tensor([[6.4719e-11, 1.0000e+00]])
tensor([[1.7494e-12, 1.0000e+00]])
tensor([[2.6931e-09, 1.0000e+00]])
tensor([[3.1971e-09, 1.0000e+00]])
tensor([[2.5471e-06, 1.0000e+00]])
tensor([[1.1374e-06, 1.0000e+00]])
tensor([[1.0000e+00, 2.3387e-34]])
tensor([[0.6583, 0.3417]])
tensor([[5.9638e-04, 9.9940e-01]])
tensor([[7.5010e-05, 9.9993e-01]])
tensor([[0.2833, 0.7167]])
tensor([[1.4745e-04, 9.9985e-01]])
tensor([[0.0100, 0.9900]])
tensor([[0.3009, 0.6991]])
tensor([[3.9821e-06, 1.0000e+00]])
tensor([[0.0065, 0.9935]])
tensor([[3.1020e-04, 9.9969e-01]])
tensor([[0.4011, 0.5989]])
tensor([[1.0325e-05, 9.9999e-01]])
tensor([[9.9607e-15, 1.0000e+00]])
tensor([[0.0032, 0.9968]])
tensor([[4.3000e-13, 1.0000e+00]])
tensor([[5.3329e-05, 9.9995e-01]])
tensor([[1.1430e-10, 1.0000e+00]])
tensor([[2.0911e-08, 1.0000e+00]])
tensor([[1.3902e-04, 9.9986e-01]])
tensor([[3.4062e-05, 9.9997e-01]])
tensor([[5.981

Warning in <TFile::Append>: Replacing existing TH1: bkg (Potential memory leak).


In [23]:
from ROOT import TCanvas

h_sig.Scale(1./h_sig.Integral()); h_sig.SetLineColor(1); h_sig.SetStats(0)
h_bkg.Scale(1./h_bkg.Integral()); h_bkg.SetLineColor(2); h_bkg.SetStats(0)
c = TCanvas()
#c.SetLogy()
h_bkg.Draw()
h_sig.Draw("same")
c.Draw()

ZeroDivisionError: float division by zero